In [ ]:
# ------------------------------------------------ Filtering ------------------------------------------------
import trimesh
import os
import numpy as np
import cadquery as cq
import random
import math
from ocp_vscode import show_object, set_port


def check_45_degree_overhang(
    shape,
    max_overhang_angle=45,
    max_bad_group_area_mm2=3,
    ignore_bottom_height=0.5,
    temp_filename="temp_overhang_check.stl"
):


    try:
        # 1. Export CadQuery shape temporarily as STL
        cq.exporters.export(shape, temp_filename) 

        # 2. Load STL as trimesh mesh
        mesh = trimesh.load_mesh(temp_filename)

        if isinstance(mesh, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))

        # Merge equal vertices before checking face connections
        mesh.merge_vertices()

        # 3. Get mesh information
        face_normals = mesh.face_normals
        face_areas = mesh.area_faces
        face_centers = mesh.triangles_center

        total_area = mesh.area
        z_min = mesh.bounds[0][2]

        if total_area <= 0:
            raise ValueError("Total mesh area is zero, so overhang area cannot be calculated.")

        bad_area = 0 
        bad_faces = 0
        bad_face_indices = []

        # 4. Find bad overhang mesh triangles
        for i, normal in enumerate(face_normals): 
            nz = normal[2] 
            face_z = face_centers[i][2]

            # Ignore bottom faces on the build plate
            if face_z <= z_min + ignore_bottom_height:
                continue

            # Only check downward-facing faces
            if nz < 0:
                surface_angle_from_horizontal = math.degrees(math.acos(abs(nz)))
                overhang_angle = 90 - surface_angle_from_horizontal 

                if overhang_angle > max_overhang_angle:
                    bad_area += face_areas[i]
                    bad_faces += 1
                    bad_face_indices.append(i)

        # 5. Group connected bad overhang faces together
        bad_face_set = set(bad_face_indices)

        bad_face_neighbours = {}

        for face_index in bad_face_indices:
            bad_face_neighbours[face_index] = []

        for face_a, face_b in mesh.face_adjacency:
            if face_a in bad_face_set and face_b in bad_face_set:
                bad_face_neighbours[face_a].append(face_b)
                bad_face_neighbours[face_b].append(face_a)

        # 6. Find connected groups of bad faces
        visited = set()
        bad_groups = []

        for face_index in bad_face_indices:
            if face_index in visited:
                continue

            group = []
            stack = [face_index]
            visited.add(face_index)

            while stack:
                current_face = stack.pop()
                group.append(current_face)

                for neighbour in bad_face_neighbours[current_face]:
                    if neighbour not in visited:
                        visited.add(neighbour)
                        stack.append(neighbour)

            bad_groups.append(group)

        # 7. Calculate largest bad group area
        largest_bad_group_area = 0
        largest_bad_group_faces = 0

        for group in bad_groups:
            group_area = sum(face_areas[face_index] for face_index in group)

            if group_area > largest_bad_group_area:
                largest_bad_group_area = group_area
                largest_bad_group_faces = len(group)

        # 8. Clean up temporary STL file
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

        # 9. Print results
        print(f"Largest bad group area: {largest_bad_group_area:.2f} mm²")

        # 10. Accept or reject based on largest grouped bad overhang area
        if largest_bad_group_area > max_bad_group_area_mm2:
            print("Part rejected: largest grouped overhang area is too large.")
            return False
        else:
            print("Part accepted: largest grouped overhang area is within allowed limits.")
            return True

    except Exception as e:
        print(f"Overhang check failed: {e}")
        return False


def check_geometry_validity(part):

    try:
 
        # 1. Keep only the largest CadQuery solid body
        solids = part.solids().vals()

        number_of_solids = len(solids)
        print(f"Number of CadQuery solid bodies before filtering: {number_of_solids}")

        if number_of_solids == 0:
            print("Part rejected: no solid bodies found.")
            return False, part

        if number_of_solids > 1:
            largest_solid = max(solids, key=lambda solid: solid.Volume())
            largest_volume = largest_solid.Volume()

            print("Smaller separate bodies were removed.")

            part = cq.Workplane("XY").add(largest_solid)
        else:
            print("Only one CadQuery solid body found.")


        # 2. Convert remaining CadQuery solid to triangle mesh
        solid = part.val()

        vertices, faces = solid.tessellate(0.1)

        mesh = trimesh.Trimesh(
            vertices=[(v.x, v.y, v.z) for v in vertices],
            faces=faces,
            process=True
        )


        # 3. Check if the mesh is a proper closed volume
        if mesh.is_volume:
            print("Part is a proper closed volume.")
        else:
            print("Part rejected: mesh is not a proper closed volume.")
            return False, part


        # 4. Check for multiple mesh components or enclosed cavities
        components = mesh.split(only_watertight=False)

        if len(components) > 1:
            print("Part rejected: part has more than 1 mesh component or an enclosed internal cavity.")
            return False, part
        else:
            print("Part has 1 mesh component.")

        return True, part

    except Exception as e:
        print(f"Additional geometry check failed: {e}")
        return False, part


# ------------------------------------------------ Run all checks and show in OCP CAD viewer ------------------------------------------------

# 1. Run the additional geometry validity check first
# This may remove smaller separate bodies
geometry_valid, part = check_geometry_validity(part)

# 2. Run the 45-degree overhang check on the cleaned part
overhang_valid = check_45_degree_overhang(
    part,
    max_overhang_angle=45,
    max_bad_group_area_mm2=3
)

# 3. The part is only valid if both checks pass
is_valid = geometry_valid and overhang_valid

# Show original part
show_object(
    part,
    name="Part",
    options={
        "color": "gold",
        "alpha": 1.0
    },
    clear=True
)

if not is_valid:
    print("This part should be removed and a new one should be generated.")
else:
    print("This part passed all checks.")